# Battery Model Dispatch Plot

This notebook loads either real electricity prices or generated price-model data and plots RL battery dispatch against mathematical optimization together with the electricity price.

In [26]:
import os
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"  # must be before pyplot import

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "Battery_model").exists():
            return candidate
    raise FileNotFoundError("Could not find the THESIS repository root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PRICE_FILE = (
    ROOT
    / "Battery_model"
    / "Optimization_Pyomo"
    / "input_data"
    / "electricity_price.txt"
)
RL_MODEL_PATH = ROOT / "src" / "battery_model" / "battery_sac_model"
OUTPUT_FIGURE = ROOT / "notebooks" / "battery_model_dispatch_plot.png"

# Choose price source: "real" or "model".
PRICE_SOURCE = "model"
PRICE_MODEL_HOURS = 8784
PRICE_MODEL_SEED = None  # Set an integer for a reproducible stochastic path.
PRICE_MODEL_INITIAL_PRICE = None
PRICE_MODEL_START_HOUR = 0.0
PRICE_MODEL_DT_HOURS = 1.0
PRICE_MODEL_PRICE_VOLATILITY_SCALE = 50.0
PRICE_MODEL_WIND_VOLATILITY_SCALE = 1.0

# Use one day for the dispatch comparison plot.
WINDOW_START = 23
WINDOW_HOURS = 24

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

ROOT


PosixPath('/home/ashish/Documents/THESIS')

In [27]:
price_source = PRICE_SOURCE.lower()

if price_source == "real":
    price_df = pd.read_csv(
        PRICE_FILE,
        sep=r"\s+",
        header=None,
        names=["timestamp_s", "price_eur_per_mwh"],
    )
    prices_full = price_df["price_eur_per_mwh"].astype(float).to_numpy()
    GLOBAL_PRICE_SCALE = np.percentile(np.abs(prices_full), 95) + 1e-6
    PRICE_SOURCE_LABEL = f"real prices from {PRICE_FILE.relative_to(ROOT)}"
elif price_source == "model":
    from src.battery_model.price_model import load_price_model_data

    prices_full, GLOBAL_PRICE_SCALE = load_price_model_data(
        num_hours=PRICE_MODEL_HOURS,
        initial_price=PRICE_MODEL_INITIAL_PRICE,
        start_hour=PRICE_MODEL_START_HOUR,
        dt_hours=PRICE_MODEL_DT_HOURS,
        seed=PRICE_MODEL_SEED,
        wind_volatility_scale=PRICE_MODEL_WIND_VOLATILITY_SCALE,
        price_volatility_scale=PRICE_MODEL_PRICE_VOLATILITY_SCALE,
    )
    price_df = pd.DataFrame(
        {
            "timestamp_s": np.arange(len(prices_full)) * PRICE_MODEL_DT_HOURS * 3600.0,
            "price_eur_per_mwh": prices_full,
        }
    )
    PRICE_SOURCE_LABEL = (
        f"coupled wind-price model, seed={PRICE_MODEL_SEED}, "
        f"price_vol_scale={PRICE_MODEL_PRICE_VOLATILITY_SCALE}"
    )
else:
    raise ValueError('PRICE_SOURCE must be either "real" or "model".')

window_end = min(WINDOW_START + WINDOW_HOURS, len(prices_full))
prices_window = prices_full[WINDOW_START:window_end]

print(f"Loaded {len(prices_full)} hourly prices from {PRICE_SOURCE_LABEL}")
print(f"Using hours {WINDOW_START} to {window_end - 1} ({len(prices_window)} hours)")
print(f"Global price scale: {GLOBAL_PRICE_SCALE:.4f}")

price_df.iloc[WINDOW_START:window_end].head()


Generated coupled wind-price dataset length: 8784 hours
Global price scale (95th pct): 58.1929
Coupled price shock std (1 h): 4.8233 EUR/MWh
Log-wind shock std (1 h): 0.2289
Wind-price shock correlation (1 h): -0.1514
Loaded 8784 hourly prices from coupled wind-price model, seed=None, price_vol_scale=50.0
Using hours 23 to 46 (24 hours)
Global price scale: 58.1929


,timestamp_s,price_eur_per_mwh
23,82800.0,23.010805
24,86400.0,34.138279
25,90000.0,42.156195
26,93600.0,55.038438
27,97200.0,60.786850


In [28]:
def build_dispatch_frame(
    method, prices, power_kw, soc, power_actual_kw=None, control=None, rewards=None
):
    prices = np.asarray(prices, dtype=float)
    power_kw = np.asarray(power_kw, dtype=float)
    soc = np.asarray(soc, dtype=float)

    data = {
        "hour": np.arange(len(prices)),
        "price_eur_per_mwh": prices,
        "power_kw": power_kw,
        "charge_kw": np.clip(power_kw, 0.0, None),
        "discharge_kw": np.clip(-power_kw, 0.0, None),
        "soc": soc,
        "method": method,
    }
    if power_actual_kw is not None:
        data["power_actual_kw"] = np.asarray(power_actual_kw, dtype=float)
    if control is not None:
        data["control_u"] = np.asarray(control, dtype=float)
    if rewards is not None:
        data["reward"] = np.asarray(rewards, dtype=float)

    return pd.DataFrame(data)


def run_rl_dispatch(prices, price_scale):
    from stable_baselines3 import SAC

    from src.battery_model.evaluate import evaluate_model

    if not RL_MODEL_PATH.with_suffix(".zip").exists():
        raise FileNotFoundError(
            f"Missing saved RL model: {RL_MODEL_PATH.with_suffix('.zip')}"
        )

    model = SAC.load(str(RL_MODEL_PATH))
    p_cmds, power_kw, power_actual_kw, soc, rewards, prices_used = evaluate_model(
        model,
        prices,
        price_scale,
    )
    return build_dispatch_frame(
        "RL SAC",
        prices_used,
        power_kw,
        soc,
        power_actual_kw=power_actual_kw,
        control=p_cmds / 20.0,
        rewards=rewards,
    )


def run_pyomo_dispatch(prices):
    from Battery_model.Optimization_Pyomo.pyomo_battery_model_v2 import (
        get_battery_model,
    )

    result = get_battery_model(prices, verbose=False)
    if result.get("status") != "success":
        raise RuntimeError(result.get("error", "Pyomo optimization failed."))

    return build_dispatch_frame(
        "Mathematical optimization",
        prices,
        result["P"],
        result["SOC"],
        power_actual_kw=result.get("P_actual"),
        control=result.get("u"),
    )


def objective_value_eur(df):
    return float(np.sum((df["price_eur_per_mwh"] / 1000.0) * df["power_kw"]))


def summarize_dispatch(df):
    objective_eur = objective_value_eur(df)
    return pd.DataFrame(
        [
            {
                "method": df["method"].iloc[0],
                "window_hours": len(df),
                "objective_value_eur": objective_eur,
                "total_cost_eur": objective_eur,
                "charge_energy_kwh": float(df["charge_kw"].sum()),
                "discharge_energy_kwh": float(df["discharge_kw"].sum()),
                "min_soc": float(df["soc"].min()),
                "max_soc": float(df["soc"].max()),
            }
        ]
    )


In [29]:
dispatch_df = run_rl_dispatch(prices_window, GLOBAL_PRICE_SCALE)
math_dispatch_df = run_pyomo_dispatch(prices_window)

print("Prepared RL SAC dispatch for the plot.")
print("Prepared mathematical optimization dispatch for the plot.")

summary_frames = [
    summarize_dispatch(dispatch_df),
    summarize_dispatch(math_dispatch_df),
]

display(pd.concat(summary_frames, ignore_index=True))
dispatch_df.head()


Prepared RL SAC dispatch for the plot.
Prepared mathematical optimization dispatch for the plot.


,method,window_hours,objective_value_eur,total_cost_eur,charge_energy_kwh,discharge_energy_kwh,min_soc,max_soc
0,RL SAC,24,-9.619582,-9.619582,207.854567,160.957283,0.000000,0.602
1,Mathematical optimization,24,-9.777096,-9.777096,239.999997,186.102315,0.002537,0.590


,hour,price_eur_per_mwh,power_kw,charge_kw,discharge_kw,soc,method,power_actual_kw,control_u,reward
0,0,23.010805,-5.427322,0.0,5.427322,0.469848,RL SAC,-6.030336,-0.271366,0.124887
1,1,34.138279,-18.093560,0.0,18.093560,0.369329,RL SAC,-20.103955,-0.904678,0.617683
2,2,42.156197,-19.153208,0.0,19.153208,0.262922,RL SAC,-21.281342,-0.957660,0.807426
3,3,55.038437,-19.359889,0.0,19.359889,0.155367,RL SAC,-21.510988,-0.967994,1.065538
4,4,60.786850,-19.533675,0.0,19.533675,0.046846,RL SAC,-21.704083,-0.976684,1.187391


In [30]:
def plot_battery_dispatch(df, math_df, output_path=None):
    hours = df["hour"].to_numpy()
    math_hours = math_df["hour"].to_numpy()
    x_min, x_max = int(hours.min()), int(hours.max())
    x_ticks = np.arange(x_min, x_max + 1, 1)

    rl_power = df["power_kw"].to_numpy(dtype=float)
    math_power = math_df["power_kw"].to_numpy(dtype=float)
    max_abs_power = max(
        float(np.max(np.abs(rl_power))),
        float(np.max(np.abs(math_power))),
        1.0,
    )

    fig, (ax_power, ax_soc) = plt.subplots(
        2,
        1,
        figsize=(14, 6.5),
        sharex=True,
        gridspec_kw={"height_ratios": [3.0, 1.0], "hspace": 0.08},
    )

    rl_area = ax_power.fill_between(
        hours,
        0.0,
        rl_power,
        step="post",
        color="#78a984",
        edgecolor="#0b5d1e",
        linewidth=1.0,
        alpha=0.72,
        label="RL charge/discharge",
    )
    ax_power.step(hours, rl_power, where="post", color="#0b5d1e", linewidth=1.2)
    math_line = ax_power.step(
        math_hours,
        math_power,
        where="post",
        color="blue",
        linewidth=2.2,
        linestyle="--",
        label="Math charge/discharge",
    )[0]
    ax_power.axhline(0.0, color="#222222", linewidth=1.0)
    ax_power.set_ylim(-max_abs_power * 1.25, max_abs_power * 1.25)
    ax_power.set_xlim(x_min, x_max)
    ax_power.set_ylabel("Power [kW]", fontweight="bold")
    ax_power.grid(True, alpha=0.28)
    ax_power.tick_params(labelbottom=False)

    ax_price = ax_power.twinx()
    price_line = ax_price.plot(
        hours,
        df["price_eur_per_mwh"],
        color="red",
        linewidth=2.2,
        label="Electricity price",
    )[0]
    ax_price.set_ylabel("Electricity price [EUR/MWh]", fontweight="bold")

    rl_soc_line = ax_soc.plot(
        hours,
        df["soc"] * 100.0,
        color="#222222",
        linewidth=2.1,
        label="RL SOC",
    )[0]
    ax_soc.fill_between(hours, 0, df["soc"] * 100.0, color="#222222", alpha=0.08)
    math_soc_line = ax_soc.plot(
        math_hours,
        math_df["soc"] * 100.0,
        color="#5f5f5f",
        linewidth=2.1,
        linestyle="--",
        label="Math SOC",
    )[0]
    ax_soc.set_ylim(0, 100)
    ax_soc.set_xlim(x_min, x_max)
    ax_soc.set_xticks(x_ticks)
    ax_soc.set_xlabel("Time [h]")
    ax_soc.set_ylabel("SOC [%]", fontweight="bold")
    ax_soc.grid(True, alpha=0.28)

    title = f"Battery dispatch: RL vs mathematical optimization ({PRICE_SOURCE}) | hours {WINDOW_START}-{window_end - 1}"
    objective_text = (
        f"RL objective: {objective_value_eur(df):.2f} EUR"
        f"  |  Math objective: {objective_value_eur(math_df):.2f} EUR"
    )
    fig.suptitle(title, fontsize=16, y=0.985)
    fig.text(0.5, 0.925, objective_text, ha="center", va="center", fontsize=11)
    fig.legend(
        [rl_area, math_line, price_line, rl_soc_line, math_soc_line],
        [
            "RL charge/discharge",
            "Math charge/discharge",
            "Electricity price",
            "RL SOC",
            "Math SOC",
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.875),
        ncol=5,
        frameon=True,
        fontsize=10,
    )
    fig.subplots_adjust(top=0.74)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to {output_path.relative_to(ROOT)}")

    return fig


plot_battery_dispatch(dispatch_df, math_dispatch_df, OUTPUT_FIGURE)

Saved figure to notebooks/battery_model_dispatch_plot.png


<Figure size 1680x780 with 3 Axes>

Set `PRICE_SOURCE = "real"` to use the input price file or `PRICE_SOURCE = "model"` to use the stochastic price model. The plot always compares RL SAC against the mathematical optimizer. Pyomo requires a working IPOPT installation.